# Twitter Sentiment Analysis Using RNN

Week 31 Graded Mini Project  
Advanced Certificate Programme in Applied Artificial Intelligence and Machine Learning

## Objective

Build an end-to-end sentiment analysis pipeline for Twitter posts using a Recurrent Neural Network (RNN). The final model should classify tweets into one of three sentiment classes: `Negative`, `Neutral`, or `Positive`.

## Assignment And Rubric Checklist

This notebook is organized to match the project task list and rubric:

1. Load the dataset.
2. Clean and preprocess tweet text.
3. Engineer numerical text features and token sequences.
4. Conduct exploratory data analysis.
5. Build an embedding-based LSTM or GRU model.
6. Train, validate, and test the model.
7. Evaluate accuracy, precision, recall, and F1-score.
8. Plot learning curves and confusion matrix.
9. Improve the model through controlled hyperparameter experiments.
10. Present final findings and sample predictions.

## Dataset Notes

The source CSV file has no header row. It will be loaded with these explicit column names:

- `tweet_id`
- `entity`
- `sentiment`
- `tweet_text`

Initial profiling found four sentiment labels in the raw dataset: `Negative`, `Positive`, `Neutral`, and `Irrelevant`. Because the assignment objective and rubric describe a three-class sentiment task, the main modeling pipeline will document and exclude `Irrelevant`, then train on `Negative`, `Neutral`, and `Positive`.

## Phase 1: Project Setup

Status: scaffold created.

The implementation will use a reproducible project structure:

- `requirements.txt` for dependencies.
- `PROJECT_CONTEXT.md` for assignment, rubric, and dataset decisions.
- `outputs/data` for cleaned working datasets.
- `outputs/figures` for generated charts.
- `outputs/models` for trained model files.
- `outputs/tables` for evaluation tables and intermediate summaries.
- `outputs/reports` for report or presentation-ready exports.

In [ ]:
# Phase 1 setup cell. Execute this before later phases.
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "twitter_training.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"
REPORTS_DIR = OUTPUT_DIR / "reports"

for directory in [FIGURES_DIR, MODELS_DIR, TABLES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path exists: {DATA_PATH.exists()} -> {DATA_PATH}")

---

# Part 1: Data Processing

## Phase 2: Loading And Raw Data Validation

Phase 2 loads the raw CSV exactly as supplied and validates its structure before any cleaning is performed. This gives the project an auditable baseline for the rubric item `Loading the Dataset` and prepares data quality evidence for the next phase.

Work completed in this phase:

- Load `twitter_training.csv` into a pandas DataFrame with explicit column names.
- Confirm shape, columns, data types, and sample rows.
- Check missing values, blank tweet text, duplicate rows, and sentiment labels.
- Compute raw sentiment distribution and text length statistics.
- Save validation summaries to `outputs/tables` for documentation and later reporting.

No cleaning is applied in this phase. Cleaning starts in Phase 3.

In [ ]:
# Phase 2 imports and constants.
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 20)

RAW_COLUMNS = ["tweet_id", "entity", "sentiment", "tweet_text"]
ASSIGNMENT_SENTIMENTS = {"Negative", "Neutral", "Positive"}

print("Phase 2 constants ready.")
print(f"Expected raw columns: {RAW_COLUMNS}")

In [ ]:
# Load the no-header CSV using explicit column names.
df_raw = pd.read_csv(
    DATA_PATH,
    header=None,
    names=RAW_COLUMNS,
    encoding="utf-8",
)

print(f"Dataset loaded successfully: {DATA_PATH.name}")
print(f"Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
print("Columns:", list(df_raw.columns))

display(df_raw.head())

In [ ]:
# Structural validation: schema, data types, missing values, and unique counts.
schema_is_valid = list(df_raw.columns) == RAW_COLUMNS

dataset_overview = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "column_count",
            "schema_matches_expected_columns",
            "unique_tweet_ids",
            "unique_entities",
            "unique_sentiments",
            "missing_tweet_text",
            "exact_duplicate_rows",
        ],
        "value": [
            len(df_raw),
            df_raw.shape[1],
            schema_is_valid,
            df_raw["tweet_id"].nunique(dropna=True),
            df_raw["entity"].nunique(dropna=True),
            df_raw["sentiment"].nunique(dropna=True),
            df_raw["tweet_text"].isna().sum(),
            df_raw.duplicated().sum(),
        ],
    }
)

missing_summary = pd.DataFrame(
    {
        "missing_count": df_raw.isna().sum(),
        "missing_percent": (df_raw.isna().mean() * 100).round(2),
        "unique_values": df_raw.nunique(dropna=True),
        "dtype": df_raw.dtypes.astype(str),
    }
).reset_index(names="column")

display(dataset_overview)
display(missing_summary)

In [ ]:
# Sentiment validation.
sentiment_distribution = (
    df_raw["sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="count")
)
sentiment_distribution["percent"] = (
    sentiment_distribution["count"] / len(df_raw) * 100
).round(2)

observed_sentiments = set(df_raw["sentiment"].dropna().unique())
extra_sentiments = sorted(observed_sentiments - ASSIGNMENT_SENTIMENTS)
missing_assignment_sentiments = sorted(ASSIGNMENT_SENTIMENTS - observed_sentiments)

sentiment_validation = pd.DataFrame(
    {
        "check": [
            "assignment_sentiments_present",
            "extra_labels_in_raw_data",
            "recommended_main_model_scope",
        ],
        "result": [
            len(missing_assignment_sentiments) == 0,
            ", ".join(extra_sentiments) if extra_sentiments else "None",
            "Use Negative, Neutral, and Positive; document and exclude Irrelevant before modeling.",
        ],
    }
)

display(sentiment_distribution)
display(sentiment_validation)

In [ ]:
# Duplicate and blank-text validation.
text_as_string = df_raw["tweet_text"].fillna("").astype(str)
blank_or_missing_text = text_as_string.str.strip().eq("")

duplicate_summary = pd.DataFrame(
    {
        "metric": [
            "blank_or_missing_tweet_text_rows",
            "exact_duplicate_rows",
            "duplicate_tweet_text_rows_non_null",
            "duplicate_tweet_text_and_sentiment_rows_non_null",
        ],
        "value": [
            int(blank_or_missing_text.sum()),
            int(df_raw.duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), "tweet_text"].duplicated().sum()),
            int(df_raw.loc[df_raw["tweet_text"].notna(), ["tweet_text", "sentiment"]].duplicated().sum()),
        ],
    }
)

blank_text_by_sentiment = (
    df_raw.loc[blank_or_missing_text, "sentiment"]
    .value_counts(dropna=False)
    .rename_axis("sentiment")
    .reset_index(name="blank_or_missing_rows")
)

display(duplicate_summary)
display(blank_text_by_sentiment)

In [ ]:
# Raw tweet length statistics. These help set later sequence length choices.
df_raw_profile = df_raw.copy()
df_raw_profile["tweet_text_str"] = text_as_string
df_raw_profile["char_len"] = df_raw_profile["tweet_text_str"].str.len()
df_raw_profile["word_len"] = df_raw_profile["tweet_text_str"].str.split().str.len()

length_summary_overall = (
    df_raw_profile[["char_len", "word_len"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
)

length_summary_by_sentiment = (
    df_raw_profile
    .groupby("sentiment")[["char_len", "word_len"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
length_summary_by_sentiment.columns = [
    f"{feature}_{stat}" for feature, stat in length_summary_by_sentiment.columns.to_flat_index()
]

top_entities_raw = (
    df_raw["entity"]
    .value_counts()
    .rename_axis("entity")
    .reset_index(name="count")
)
top_entities_raw["percent"] = (top_entities_raw["count"] / len(df_raw) * 100).round(2)

display(length_summary_overall)
display(length_summary_by_sentiment)
display(top_entities_raw.head(15))

In [ ]:
# Save Phase 2 audit tables for the report and later phases.
phase2_tables = {
    "phase2_dataset_overview.csv": dataset_overview,
    "phase2_missing_summary.csv": missing_summary,
    "phase2_sentiment_distribution.csv": sentiment_distribution,
    "phase2_sentiment_validation.csv": sentiment_validation,
    "phase2_duplicate_summary.csv": duplicate_summary,
    "phase2_blank_text_by_sentiment.csv": blank_text_by_sentiment,
    "phase2_length_summary_overall.csv": length_summary_overall.reset_index(names="statistic"),
    "phase2_length_summary_by_sentiment.csv": length_summary_by_sentiment.reset_index(),
    "phase2_top_entities_raw.csv": top_entities_raw,
}

for filename, table in phase2_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

phase2_summary_lines = [
    "# Phase 2 Validation Summary",
    "",
    f"- Dataset rows: {len(df_raw):,}",
    f"- Dataset columns: {df_raw.shape[1]:,}",
    f"- Schema matches expected columns: {schema_is_valid}",
    f"- Missing tweet_text values: {int(df_raw['tweet_text'].isna().sum()):,}",
    f"- Blank or missing tweet_text rows: {int(blank_or_missing_text.sum()):,}",
    f"- Exact duplicate rows: {int(df_raw.duplicated().sum()):,}",
    f"- Unique entities: {df_raw['entity'].nunique(dropna=True):,}",
    f"- Unique sentiment labels: {df_raw['sentiment'].nunique(dropna=True):,}",
    f"- Extra raw label outside assignment scope: {', '.join(extra_sentiments) if extra_sentiments else 'None'}",
    "",
    "## Raw Sentiment Distribution",
    "",
]
phase2_summary_lines.extend(
    [
        f"- {row.sentiment}: {int(row.count):,} ({row.percent:.2f}%)"
        for row in sentiment_distribution.itertuples(index=False)
    ]
)
phase2_summary_lines.extend(
    [
        "",
        "## Recommendation For Phase 3",
        "",
        "Create a cleaned working DataFrame by dropping blank or missing tweet text, removing exact duplicate rows, documenting exclusion of Irrelevant, and then applying tweet text cleaning.",
    ]
)
summary_path = REPORTS_DIR / "phase2_validation_summary.md"
summary_path.write_text("\n".join(phase2_summary_lines) + "\n", encoding="utf-8")

print(f"Saved {len(phase2_tables)} Phase 2 tables to {TABLES_DIR}")
print(f"Saved Phase 2 validation summary to {summary_path}")

## Phase 2 Findings To Carry Forward

The raw dataset loads successfully into four columns: `tweet_id`, `entity`, `sentiment`, and `tweet_text`.

Important findings to use in Phase 3:

- The CSV has no header row, so explicit column names are required.
- The raw dataset includes a fourth label, `Irrelevant`, even though the assignment asks for `Negative`, `Neutral`, and `Positive` sentiment classification.
- Blank or missing tweet text and duplicate rows must be handled before modeling.
- Text length statistics should guide sequence padding choices in the RNN phase.

Recommended Phase 3 action: create a cleaned working DataFrame that drops blank or missing tweet text, removes exact duplicate rows, documents the exclusion of `Irrelevant`, and prepares text-cleaning functions for URLs, mentions, hashtags, special characters, tokenization, stop-word removal, and lemmatization or stemming.

## Phase 3: Data Cleaning

Phase 3 creates the cleaned working dataset for downstream EDA, feature engineering, and RNN modeling.

Cleaning decisions:

- Keep the original `twitter_training.csv` unchanged.
- Drop missing or blank tweet text because those rows cannot provide sentiment signal.
- Remove exact duplicate rows to reduce repeated examples and reporting distortion.
- Exclude `Irrelevant` from the main dataset because the assignment rubric targets `Negative`, `Neutral`, and `Positive` sentiment classification.
- Create `model_text` after lowercasing and removing URLs, mentions, hashtags, special characters, and extra whitespace.
- Create `processed_text` after tokenization, stop-word removal, and stemming. Negation words such as `no`, `not`, and `never` are intentionally preserved because they are important for sentiment.
- Drop rows that become empty after text cleaning.

The cleaned dataset is saved under `outputs/data` so later phases can load a stable working file.

In [ ]:
# Phase 3 directories and cleaning resources.
import html
import re

DATA_OUTPUT_DIR = OUTPUT_DIR / "data"
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This compact stop-word list is kept local so the notebook does not depend on downloading
# external NLTK corpora. Negation terms are deliberately excluded from the list.
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him", "his",
    "i", "if", "in", "into", "is", "it", "its", "me", "my", "of", "on",
    "or", "our", "ours", "she", "so", "than", "that", "the", "their", "them",
    "they", "this", "to", "was", "we", "were", "will", "with", "you", "your",
    "yours", "all", "do", "does", "did", "just", "can", "could", "should",
    "would", "about", "above", "after", "again", "against", "am", "any",
    "because", "before", "below", "between", "both", "down", "during", "each",
    "few", "further", "here", "how", "more", "most", "other", "over", "own",
    "same", "some", "such", "then", "there", "these", "those", "through", "too",
    "under", "until", "up", "very", "what", "when", "where", "which", "who",
    "whom", "why", "s", "t", "now", "get", "got", "im", "ll", "re", "ve",
    "u", "ur", "rt"
}
NEGATION_WORDS = {"no", "not", "nor", "never", "none", "cannot", "cant", "dont", "wont"}
STOP_WORDS = STOP_WORDS - NEGATION_WORDS

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
NON_LETTER_RE = re.compile(r"[^a-z\s]")
WHITESPACE_RE = re.compile(r"\s+")

try:
    from nltk.stem import PorterStemmer
    STEMMER = PorterStemmer()
    STEMMER_NAME = "nltk_porter"
except ImportError:
    STEMMER = None
    STEMMER_NAME = "fallback_suffix_stemmer"

print(f"Phase 3 cleaning resources ready. Stemming method: {STEMMER_NAME}")

In [ ]:
# Text cleaning helper functions.
def expand_common_contractions(text):
    """Normalize common English contractions before punctuation removal."""
    replacements = {
        "can't": "can not",
        "cannot": "can not",
        "won't": "will not",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'d": " would",
        "'m": " am",
        "'s": " ",
    }
    for pattern, replacement in replacements.items():
        text = text.replace(pattern, replacement)
    return text


def basic_clean_tweet(text):
    """Lowercase and remove Twitter-specific noise and non-letter characters."""
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text).lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = HASHTAG_RE.sub(" ", text)
    text = expand_common_contractions(text)
    text = NON_LETTER_RE.sub(" ", text)
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text


def tokenize_text(text):
    return [token for token in str(text).split() if token]


def remove_stop_words(tokens):
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 1]


def fallback_stem(word):
    """Small deterministic fallback used only when NLTK is unavailable."""
    if len(word) <= 4:
        return word
    for suffix, replacement in [
        ("ingly", ""),
        ("edly", ""),
        ("ing", ""),
        ("ies", "y"),
        ("ied", "y"),
        ("ed", ""),
        ("ly", ""),
        ("s", ""),
    ]:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: -len(suffix)] + replacement
    return word


def stem_tokens(tokens):
    if STEMMER is not None:
        return [STEMMER.stem(token) for token in tokens]
    return [fallback_stem(token) for token in tokens]


print("Phase 3 helper functions ready.")

In [ ]:
# Build the cleaned working dataset.
df_clean = df_raw.copy()
raw_row_count = len(df_clean)

df_clean["tweet_text_stripped"] = df_clean["tweet_text"].fillna("").astype(str).str.strip()
blank_or_missing_mask = df_clean["tweet_text_stripped"].eq("")
blank_or_missing_removed = int(blank_or_missing_mask.sum())
df_clean = df_clean.loc[~blank_or_missing_mask].copy()

before_duplicate_drop = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=RAW_COLUMNS).copy()
duplicate_rows_removed = before_duplicate_drop - len(df_clean)

before_label_filter = len(df_clean)
df_clean = df_clean.loc[df_clean["sentiment"].isin(sorted(ASSIGNMENT_SENTIMENTS))].copy()
irrelevant_rows_removed = before_label_filter - len(df_clean)

df_clean["model_text"] = df_clean["tweet_text"].apply(basic_clean_tweet)
empty_after_cleaning_mask = df_clean["model_text"].str.strip().eq("")
empty_after_cleaning_removed = int(empty_after_cleaning_mask.sum())
df_clean = df_clean.loc[~empty_after_cleaning_mask].copy()

df_clean["tokens"] = df_clean["model_text"].apply(tokenize_text)
df_clean["tokens_no_stopwords"] = df_clean["tokens"].apply(remove_stop_words)
df_clean["stemmed_tokens"] = df_clean["tokens_no_stopwords"].apply(stem_tokens)
df_clean["processed_text"] = df_clean["stemmed_tokens"].apply(" ".join)

df_clean["raw_char_len"] = df_clean["tweet_text"].astype(str).str.len()
df_clean["raw_word_len"] = df_clean["tweet_text"].astype(str).str.split().str.len()
df_clean["model_word_len"] = df_clean["tokens"].apply(len)
df_clean["processed_word_len"] = df_clean["stemmed_tokens"].apply(len)

# Convert token lists to space-separated strings for inspection, then export only
# stable downstream columns to keep the cleaned artifact compact.
df_clean_output = df_clean.copy()
df_clean_output["tokens"] = df_clean_output["tokens"].apply(" ".join)
df_clean_output["tokens_no_stopwords"] = df_clean_output["tokens_no_stopwords"].apply(" ".join)
df_clean_output["stemmed_tokens"] = df_clean_output["stemmed_tokens"].apply(" ".join)
cleaned_export_columns = [
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "raw_char_len",
    "raw_word_len",
    "model_word_len",
    "processed_word_len",
]
df_clean_export = df_clean_output[cleaned_export_columns].copy()

cleaned_dataset_path = DATA_OUTPUT_DIR / "twitter_training_cleaned_phase3.csv"
df_clean_export.to_csv(cleaned_dataset_path, index=False)

print(f"Raw rows: {raw_row_count:,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Cleaned dataset saved to: {cleaned_dataset_path}")
display(df_clean_export[["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "processed_text"]].head())

In [ ]:
# Phase 3 cleaning audit tables.
cleaning_steps = pd.DataFrame(
    {
        "step": [
            "raw_rows",
            "drop_blank_or_missing_tweet_text",
            "drop_exact_duplicate_rows",
            "exclude_irrelevant_label",
            "drop_empty_text_after_cleaning",
            "final_cleaned_rows",
        ],
        "rows_removed": [
            0,
            blank_or_missing_removed,
            duplicate_rows_removed,
            irrelevant_rows_removed,
            empty_after_cleaning_removed,
            0,
        ],
        "rows_remaining": [
            raw_row_count,
            raw_row_count - blank_or_missing_removed,
            raw_row_count - blank_or_missing_removed - duplicate_rows_removed,
            raw_row_count - blank_or_missing_removed - duplicate_rows_removed - irrelevant_rows_removed,
            len(df_clean),
            len(df_clean),
        ],
    }
)

phase3_sentiment_distribution = (
    df_clean["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)
phase3_sentiment_distribution["percent"] = (
    phase3_sentiment_distribution["count"] / len(df_clean) * 100
).round(2)

phase3_length_summary = (
    df_clean[["raw_word_len", "model_word_len", "processed_word_len"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
    .reset_index(names="statistic")
)

phase3_text_quality_summary = pd.DataFrame(
    {
        "metric": [
            "rows_with_urls_in_raw_text",
            "rows_with_mentions_in_raw_text",
            "rows_with_hashtags_in_raw_text",
            "rows_with_empty_model_text_after_cleaning_removed",
            "rows_with_empty_processed_text_remaining",
            "stemming_method",
        ],
        "value": [
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(URL_RE).sum()),
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(MENTION_RE).sum()),
            int(df_raw["tweet_text"].fillna("").astype(str).str.contains(HASHTAG_RE).sum()),
            empty_after_cleaning_removed,
            int(df_clean["processed_text"].str.strip().eq("").sum()),
            STEMMER_NAME,
        ],
    }
)

phase3_sample_cleaned_rows = df_clean_export[
    ["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "processed_text"]
].sample(12, random_state=42)

phase3_tables = {
    "phase3_cleaning_steps.csv": cleaning_steps,
    "phase3_cleaned_sentiment_distribution.csv": phase3_sentiment_distribution,
    "phase3_length_summary.csv": phase3_length_summary,
    "phase3_text_quality_summary.csv": phase3_text_quality_summary,
    "phase3_sample_cleaned_rows.csv": phase3_sample_cleaned_rows,
}

for filename, table in phase3_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

display(cleaning_steps)
display(phase3_sentiment_distribution)
display(phase3_text_quality_summary)

In [ ]:
# Save a concise Phase 3 report summary.
phase3_summary_lines = [
    "# Phase 3 Data Cleaning Summary",
    "",
    f"- Raw rows loaded: {raw_row_count:,}",
    f"- Removed blank or missing tweet text rows: {blank_or_missing_removed:,}",
    f"- Removed exact duplicate rows after blank-text removal: {duplicate_rows_removed:,}",
    f"- Removed rows with Irrelevant label: {irrelevant_rows_removed:,}",
    f"- Removed rows that became empty after text cleaning: {empty_after_cleaning_removed:,}",
    f"- Final cleaned rows: {len(df_clean):,}",
    f"- Stemming method used in this run: {STEMMER_NAME}",
    f"- Cleaned dataset: {cleaned_dataset_path}",
    "",
    "## Final Cleaned Sentiment Distribution",
    "",
]
phase3_summary_lines.extend(
    [
        f"- {row.sentiment}: {int(row.count):,} ({row.percent:.2f}%)"
        for row in phase3_sentiment_distribution.itertuples(index=False)
    ]
)
phase3_summary_lines.extend(
    [
        "",
        "## Cleaning Notes",
        "",
        "The original raw CSV was not modified. The cleaned dataset excludes Irrelevant because the graded assignment defines a three-class sentiment task. The notebook keeps both model_text and processed_text so later RNN work can use a sequence-friendly text representation while EDA and rubric documentation can show stop-word removal and stemming.",
    ]
)

phase3_summary_path = REPORTS_DIR / "phase3_cleaning_summary.md"
phase3_summary_path.write_text("\n".join(phase3_summary_lines) + "\n", encoding="utf-8")

print(f"Saved Phase 3 tables to {TABLES_DIR}")
print(f"Saved Phase 3 summary to {phase3_summary_path}")

## Phase 3 Findings To Carry Forward

The cleaned dataset is now ready for EDA and feature engineering.

Carry-forward points:

- `model_text` should be used for RNN sequence modeling because it preserves more word order and context.
- `processed_text` and `stemmed_tokens` are useful for rubric-visible preprocessing, top-word analysis, and word clouds.
- The final model should remain a three-class classifier: `Negative`, `Neutral`, and `Positive`.
- Later train/test splits should be created from the cleaned dataset, not the raw CSV.
- Sequence length decisions should use the cleaned `model_word_len` distribution rather than raw tweet length alone.

## Phase 4: Text Preprocessing

Phase 4 converts the cleaned text fields into stable text-processing artifacts for EDA and later feature engineering.

Scope of this phase:

- Load the Phase 3 cleaned dataset.
- Validate that only `Negative`, `Neutral`, and `Positive` labels remain.
- Create `model_tokens` from `model_text` for later RNN sequence construction.
- Create `analysis_text` and `analysis_tokens` for EDA. `analysis_text` uses `processed_text` where available, and falls back to `model_text` when stop-word removal and stemming leave a row empty.
- Save token-count summaries, top token frequency tables, label mapping metadata, and a compact preprocessed dataset.

This phase does not create padded integer sequences, TF-IDF matrices, or embeddings. Those belong to the Feature Engineering phase.

In [ ]:
# Phase 4: load the cleaned dataset created in Phase 3.
from collections import Counter

PHASE3_CLEANED_PATH = DATA_OUTPUT_DIR / "twitter_training_cleaned_phase3.csv"
PHASE4_PREPROCESSED_PATH = DATA_OUTPUT_DIR / "twitter_text_preprocessed_phase4.csv"

df_text = pd.read_csv(PHASE3_CLEANED_PATH)

required_phase3_columns = {
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "raw_word_len",
    "model_word_len",
    "processed_word_len",
}
missing_phase3_columns = sorted(required_phase3_columns - set(df_text.columns))
if missing_phase3_columns:
    raise ValueError(f"Missing expected Phase 3 columns: {missing_phase3_columns}")

remaining_labels = set(df_text["sentiment"].dropna().unique())
if remaining_labels != ASSIGNMENT_SENTIMENTS:
    raise ValueError(f"Unexpected cleaned sentiment labels: {sorted(remaining_labels)}")

print(f"Loaded Phase 3 cleaned dataset: {PHASE3_CLEANED_PATH}")
print(f"Shape: {df_text.shape[0]:,} rows x {df_text.shape[1]:,} columns")
print(f"Labels: {sorted(remaining_labels)}")

In [ ]:
# Create reusable text and token fields for later phases.
df_preprocessed = df_text.copy()

df_preprocessed["model_text"] = df_preprocessed["model_text"].fillna("").astype(str).str.strip()
df_preprocessed["processed_text"] = df_preprocessed["processed_text"].fillna("").astype(str).str.strip()

empty_processed_text_before_fallback = int(df_preprocessed["processed_text"].eq("").sum())
df_preprocessed["analysis_text"] = np.where(
    df_preprocessed["processed_text"].eq(""),
    df_preprocessed["model_text"],
    df_preprocessed["processed_text"],
)

df_preprocessed["model_tokens"] = df_preprocessed["model_text"].str.split()
df_preprocessed["analysis_tokens"] = df_preprocessed["analysis_text"].str.split()
df_preprocessed["model_token_count"] = df_preprocessed["model_tokens"].apply(len)
df_preprocessed["analysis_token_count"] = df_preprocessed["analysis_tokens"].apply(len)

empty_model_tokens = int(df_preprocessed["model_token_count"].eq(0).sum())
empty_analysis_tokens = int(df_preprocessed["analysis_token_count"].eq(0).sum())
if empty_model_tokens or empty_analysis_tokens:
    raise ValueError(
        f"Unexpected empty token rows: model={empty_model_tokens}, analysis={empty_analysis_tokens}"
    )

# Store tokenized text as space-separated strings for readable CSV artifacts.
df_preprocessed_export = df_preprocessed.copy()
df_preprocessed_export["model_tokens"] = df_preprocessed_export["model_tokens"].apply(" ".join)
df_preprocessed_export["analysis_tokens"] = df_preprocessed_export["analysis_tokens"].apply(" ".join)

phase4_export_columns = [
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "processed_text",
    "analysis_text",
    "raw_word_len",
    "model_token_count",
    "analysis_token_count",
]
df_preprocessed_export = df_preprocessed_export[phase4_export_columns].copy()
df_preprocessed_export.to_csv(PHASE4_PREPROCESSED_PATH, index=False)

display(df_preprocessed[["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "analysis_text", "model_token_count", "analysis_token_count"]].head())
print(f"Preprocessed dataset saved to: {PHASE4_PREPROCESSED_PATH}")

In [ ]:
# Phase 4 summaries and reusable metadata.
label_order = ["Negative", "Neutral", "Positive"]
label_mapping = pd.DataFrame(
    {
        "sentiment": label_order,
        "label_id": list(range(len(label_order))),
        "note": ["negative class", "neutral class", "positive class"],
    }
)

model_p99 = float(df_preprocessed["model_token_count"].quantile(0.99))
recommended_max_sequence_length = int(np.ceil(model_p99 / 10) * 10)

phase4_overview = pd.DataFrame(
    {
        "metric": [
            "rows_preprocessed",
            "unique_entities",
            "unique_sentiments",
            "empty_processed_text_before_analysis_fallback",
            "empty_model_token_rows",
            "empty_analysis_token_rows",
            "model_token_count_p95",
            "model_token_count_p99",
            "recommended_max_sequence_length_for_later_padding",
        ],
        "value": [
            f"{len(df_preprocessed):,}",
            f"{df_preprocessed['entity'].nunique():,}",
            f"{df_preprocessed['sentiment'].nunique():,}",
            f"{empty_processed_text_before_fallback:,}",
            f"{empty_model_tokens:,}",
            f"{empty_analysis_tokens:,}",
            f"{float(df_preprocessed['model_token_count'].quantile(0.95)):.2f}",
            f"{model_p99:.2f}",
            f"{recommended_max_sequence_length:,}",
        ],
    }
)

phase4_token_length_summary = (
    df_preprocessed[["model_token_count", "analysis_token_count"]]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .round(2)
    .reset_index(names="statistic")
)

phase4_token_length_by_sentiment = (
    df_preprocessed
    .groupby("sentiment")[["model_token_count", "analysis_token_count"]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)
phase4_token_length_by_sentiment.columns = [
    f"{feature}_{stat}" for feature, stat in phase4_token_length_by_sentiment.columns.to_flat_index()
]
phase4_token_length_by_sentiment = phase4_token_length_by_sentiment.reset_index()

model_text_sentiment_duplicates = int(df_preprocessed.duplicated(["model_text", "sentiment"]).sum())
model_text_duplicates = int(df_preprocessed.duplicated(["model_text"]).sum())
analysis_text_sentiment_duplicates = int(df_preprocessed.duplicated(["analysis_text", "sentiment"]).sum())
model_text_label_counts = df_preprocessed.groupby("model_text")["sentiment"].nunique()
conflicting_model_texts = model_text_label_counts[model_text_label_counts > 1].index
conflicting_model_text_count = len(conflicting_model_texts)
rows_in_conflicting_model_text_groups = int(
    df_preprocessed["model_text"].isin(conflicting_model_texts).sum()
)

phase4_duplicate_text_audit = pd.DataFrame(
    {
        "metric": [
            "duplicate_model_text_rows",
            "duplicate_model_text_and_sentiment_rows",
            "duplicate_analysis_text_and_sentiment_rows",
            "model_text_values_with_multiple_sentiment_labels",
            "rows_in_conflicting_model_text_groups",
        ],
        "value": [
            model_text_duplicates,
            model_text_sentiment_duplicates,
            analysis_text_sentiment_duplicates,
            conflicting_model_text_count,
            rows_in_conflicting_model_text_groups,
        ],
        "phase5_or_phase6_action": [
            "Audit before EDA; repeated cleaned text can dominate frequency counts.",
            "Consider deduplicating text-label pairs inside the modeling dataset if leakage risk is high.",
            "Use with care for EDA only; analysis_text is more aggressively reduced than model_text.",
            "Inspect before modeling because the same cleaned text carries multiple labels.",
            "Avoid random leakage by handling or grouping these cases before final train/test split.",
        ],
    }
)

phase4_conflicting_model_text_examples = (
    df_preprocessed.loc[
        df_preprocessed["model_text"].isin(conflicting_model_texts),
        ["tweet_id", "entity", "sentiment", "tweet_text", "model_text", "analysis_text"],
    ]
    .sort_values(["model_text", "sentiment", "tweet_id"])
    .head(60)
)

display(phase4_overview)
display(label_mapping)
display(phase4_token_length_summary)
display(phase4_duplicate_text_audit)

In [ ]:
# Token frequency tables for EDA readiness. These are descriptive, not model vocabularies.
def token_frequency_table(token_series, top_n=100):
    counter = Counter()
    for tokens in token_series:
        counter.update(tokens)
    total_tokens = sum(counter.values())
    rows = [
        {
            "token": token,
            "count": count,
            "percent_of_tokens": round(count / total_tokens * 100, 4) if total_tokens else 0,
        }
        for token, count in counter.most_common(top_n)
    ]
    return pd.DataFrame(rows)


phase4_top_analysis_tokens = token_frequency_table(df_preprocessed["analysis_tokens"], top_n=100)
phase4_top_model_tokens = token_frequency_table(df_preprocessed["model_tokens"], top_n=100)

by_sentiment_frames = []
for sentiment, group in df_preprocessed.groupby("sentiment"):
    table = token_frequency_table(group["analysis_tokens"], top_n=50)
    table.insert(0, "sentiment", sentiment)
    table.insert(1, "rank", range(1, len(table) + 1))
    by_sentiment_frames.append(table)

phase4_top_analysis_tokens_by_sentiment = pd.concat(by_sentiment_frames, ignore_index=True)

display(phase4_top_analysis_tokens.head(20))
display(phase4_top_analysis_tokens_by_sentiment.head(15))

In [ ]:
# Save Phase 4 artifacts for EDA and later feature engineering.
phase4_sample_rows = df_preprocessed_export[
    ["tweet_id", "entity", "sentiment", "model_text", "analysis_text", "model_token_count", "analysis_token_count"]
].sample(12, random_state=42)

phase4_tables = {
    "phase4_preprocessing_overview.csv": phase4_overview,
    "phase4_label_mapping.csv": label_mapping,
    "phase4_token_length_summary.csv": phase4_token_length_summary,
    "phase4_token_length_by_sentiment.csv": phase4_token_length_by_sentiment,
    "phase4_top_analysis_tokens.csv": phase4_top_analysis_tokens,
    "phase4_top_model_tokens.csv": phase4_top_model_tokens,
    "phase4_top_analysis_tokens_by_sentiment.csv": phase4_top_analysis_tokens_by_sentiment,
    "phase4_duplicate_text_audit.csv": phase4_duplicate_text_audit,
    "phase4_conflicting_model_text_examples.csv": phase4_conflicting_model_text_examples,
    "phase4_sample_preprocessed_rows.csv": phase4_sample_rows,
}

for filename, table in phase4_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

phase4_summary_lines = [
    "# Phase 4 Text Preprocessing Summary",
    "",
    f"- Source cleaned dataset: {PHASE3_CLEANED_PATH}",
    f"- Preprocessed dataset: {PHASE4_PREPROCESSED_PATH}",
    f"- Rows preprocessed: {len(df_preprocessed):,}",
    f"- Empty processed_text rows handled with model_text fallback: {empty_processed_text_before_fallback:,}",
    f"- Empty model token rows after preprocessing: {empty_model_tokens:,}",
    f"- Empty analysis token rows after preprocessing: {empty_analysis_tokens:,}",
    f"- Model token count p95: {float(df_preprocessed['model_token_count'].quantile(0.95)):.2f}",
    f"- Model token count p99: {model_p99:.2f}",
    f"- Recommended max sequence length for later padding: {recommended_max_sequence_length}",
    f"- Duplicate model_text and sentiment rows to audit before modeling: {model_text_sentiment_duplicates:,}",
    f"- Cleaned model_text values with multiple sentiment labels: {conflicting_model_text_count:,}",
    f"- Rows in conflicting model_text groups: {rows_in_conflicting_model_text_groups:,}",
    "",
    "## Label Mapping For Later Modeling",
    "",
]
phase4_summary_lines.extend(
    [f"- {row.sentiment}: {row.label_id}" for row in label_mapping.itertuples(index=False)]
)
phase4_summary_lines.extend(
    [
        "",
        "## Notes",
        "",
        "The Phase 4 token frequency tables are descriptive preprocessing artifacts for EDA. They are not the final model vocabulary. The model vocabulary must be built from the training split only during feature engineering to avoid data leakage.",
        "",
        "The duplicate-text audit is intentionally not applied as a row-removal step in this phase. It should guide Phase 6 train/test splitting and optional text-label deduplication so repeated cleaned text does not leak across splits.",
    ]
)

phase4_summary_path = REPORTS_DIR / "phase4_text_preprocessing_summary.md"
phase4_summary_path.write_text("\n".join(phase4_summary_lines) + "\n", encoding="utf-8")

print(f"Saved preprocessed dataset to {PHASE4_PREPROCESSED_PATH}")
print(f"Saved {len(phase4_tables)} Phase 4 tables to {TABLES_DIR}")
print(f"Saved Phase 4 summary to {phase4_summary_path}")

## Phase 4 Findings To Carry Forward

Phase 4 produces a stable preprocessed text dataset and descriptive token artifacts.

Carry-forward points:

- Use `model_text` or `model_tokens` when creating RNN token sequences in Feature Engineering.
- Use `analysis_text` and `analysis_tokens` for Phase 5 EDA word frequencies and word clouds.
- Use the saved label mapping consistently when converting sentiment labels for modeling.
- The recommended maximum sequence length is based on the cleaned `model_token_count` distribution and should be revisited only if later train/test split statistics differ materially.
- Do not treat Phase 4 frequency tables as the model vocabulary; the model vocabulary must be learned from the training split only.

# Part 2: Exploratory Data Analysis

## Phase 5: Exploratory Data Analysis

Phase 5 explores the cleaned and preprocessed three-class dataset before feature engineering and modeling.

EDA deliverables for the rubric:

- Basic statistics for the cleaned dataset.
- Sentiment distribution.
- Entity/topic distribution.
- Top words by sentiment.
- Word-cloud-style visualizations for positive and negative tweets.
- Tweet length patterns by sentiment.
- Written insights and modeling implications.

The figures in this phase are saved as SVG files under `outputs/figures`.

In [ ]:
# Phase 5: load the preprocessed dataset from Phase 4.
import math
import html as html_lib

PHASE4_PREPROCESSED_PATH = DATA_OUTPUT_DIR / "twitter_text_preprocessed_phase4.csv"
df_eda = pd.read_csv(PHASE4_PREPROCESSED_PATH)

required_phase5_columns = {
    "tweet_id",
    "entity",
    "sentiment",
    "tweet_text",
    "model_text",
    "analysis_text",
    "raw_word_len",
    "model_token_count",
    "analysis_token_count",
}
missing_phase5_columns = sorted(required_phase5_columns - set(df_eda.columns))
if missing_phase5_columns:
    raise ValueError(f"Missing expected Phase 5 columns: {missing_phase5_columns}")

if set(df_eda["sentiment"].unique()) != ASSIGNMENT_SENTIMENTS:
    raise ValueError("Phase 5 expected only Negative, Neutral, and Positive labels.")

df_eda["analysis_tokens"] = df_eda["analysis_text"].fillna("").astype(str).str.split()
df_eda["model_tokens"] = df_eda["model_text"].fillna("").astype(str).str.split()

print(f"Loaded Phase 4 preprocessed dataset: {PHASE4_PREPROCESSED_PATH}")
print(f"EDA shape: {df_eda.shape[0]:,} rows x {df_eda.shape[1]:,} columns")
print(f"Sentiment labels: {sorted(df_eda['sentiment'].unique())}")

In [ ]:
# Phase 5 summary statistics and distributions.
sentiment_distribution_eda = (
    df_eda["sentiment"]
    .value_counts()
    .reindex(["Negative", "Positive", "Neutral"])
    .rename_axis("sentiment")
    .reset_index(name="count")
)
sentiment_distribution_eda["percent"] = (
    sentiment_distribution_eda["count"] / len(df_eda) * 100
).round(2)

entity_distribution_eda = (
    df_eda["entity"]
    .value_counts()
    .rename_axis("entity")
    .reset_index(name="count")
)
entity_distribution_eda["percent"] = (entity_distribution_eda["count"] / len(df_eda) * 100).round(2)

phase5_basic_statistics = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_tweet_ids",
            "unique_entities",
            "unique_sentiments",
            "mode_sentiment",
            "mode_entity",
            "mean_raw_word_len",
            "median_raw_word_len",
            "mean_model_token_count",
            "median_model_token_count",
            "p95_model_token_count",
            "p99_model_token_count",
        ],
        "value": [
            f"{len(df_eda):,}",
            f"{df_eda.shape[1]:,}",
            f"{df_eda['tweet_id'].nunique():,}",
            f"{df_eda['entity'].nunique():,}",
            f"{df_eda['sentiment'].nunique():,}",
            df_eda["sentiment"].mode().iat[0],
            df_eda["entity"].mode().iat[0],
            f"{df_eda['raw_word_len'].mean():.2f}",
            f"{df_eda['raw_word_len'].median():.2f}",
            f"{df_eda['model_token_count'].mean():.2f}",
            f"{df_eda['model_token_count'].median():.2f}",
            f"{df_eda['model_token_count'].quantile(0.95):.2f}",
            f"{df_eda['model_token_count'].quantile(0.99):.2f}",
        ],
    }
)

length_by_sentiment_eda = (
    df_eda
    .groupby("sentiment")
    .agg(
        rows=("model_token_count", "count"),
        raw_word_mean=("raw_word_len", "mean"),
        raw_word_median=("raw_word_len", "median"),
        model_token_mean=("model_token_count", "mean"),
        model_token_median=("model_token_count", "median"),
        model_token_p25=("model_token_count", lambda values: values.quantile(0.25)),
        model_token_p75=("model_token_count", lambda values: values.quantile(0.75)),
        model_token_p90=("model_token_count", lambda values: values.quantile(0.90)),
        model_token_p95=("model_token_count", lambda values: values.quantile(0.95)),
        analysis_token_mean=("analysis_token_count", "mean"),
        analysis_token_median=("analysis_token_count", "median"),
    )
    .round(2)
    .reset_index()
)

display(phase5_basic_statistics)
display(sentiment_distribution_eda)
display(entity_distribution_eda.head(15))
display(length_by_sentiment_eda)

In [ ]:
# Phase 5 token frequencies by sentiment.
def phase5_token_frequency(token_lists, top_n=50):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    total = sum(counter.values())
    return pd.DataFrame(
        [
            {
                "token": token,
                "count": count,
                "percent_of_tokens": round(count / total * 100, 4) if total else 0,
            }
            for token, count in counter.most_common(top_n)
        ]
    )


top_tokens_by_sentiment_frames = []
top_tokens_lookup = {}
for sentiment in ["Negative", "Neutral", "Positive"]:
    sentiment_tokens = df_eda.loc[df_eda["sentiment"].eq(sentiment), "analysis_tokens"]
    token_table = phase5_token_frequency(sentiment_tokens, top_n=60)
    token_table.insert(0, "sentiment", sentiment)
    token_table.insert(1, "rank", range(1, len(token_table) + 1))
    top_tokens_by_sentiment_frames.append(token_table)
    top_tokens_lookup[sentiment] = token_table.copy()

phase5_top_tokens_by_sentiment = pd.concat(top_tokens_by_sentiment_frames, ignore_index=True)

display(phase5_top_tokens_by_sentiment.groupby("sentiment").head(12))

In [ ]:
# SVG visualization helpers. These avoid runtime dependency on plotting libraries.
def svg_text(text):
    return html_lib.escape(str(text), quote=True)


def save_svg(content, path, width, height):
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" role="img">\n{content}\n</svg>\n'
    )
    path.write_text(svg, encoding="utf-8")


def save_vertical_bar_chart(data, label_col, value_col, title, y_label, path, color="#2f6f9f", width=920, height=560):
    margin_left, margin_right, margin_top, margin_bottom = 90, 40, 80, 130
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    max_value = max(float(v) for v in data[value_col])
    bar_gap = 24
    bar_width = (plot_width - bar_gap * (len(data) - 1)) / len(data)
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="35" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">{svg_text(title)}</text>',
        f'<line x1="{margin_left}" y1="{height-margin_bottom}" x2="{width-margin_right}" y2="{height-margin_bottom}" stroke="#333" stroke-width="1"/>',
        f'<line x1="{margin_left}" y1="{margin_top}" x2="{margin_left}" y2="{height-margin_bottom}" stroke="#333" stroke-width="1"/>',
        f'<text x="25" y="{margin_top + plot_height/2}" text-anchor="middle" transform="rotate(-90 25 {margin_top + plot_height/2})" font-family="Arial" font-size="14" fill="#333">{svg_text(y_label)}</text>',
    ]
    for i, row in data.reset_index(drop=True).iterrows():
        value = float(row[value_col])
        bar_height = 0 if max_value == 0 else value / max_value * plot_height
        x = margin_left + i * (bar_width + bar_gap)
        y = height - margin_bottom - bar_height
        label = row[label_col]
        percent = row.get("percent", None)
        value_label = f"{int(value):,}"
        if percent is not None:
            value_label += f" ({float(percent):.1f}%)"
        parts.extend([
            f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" rx="4" fill="{color}"/>',
            f'<text x="{x + bar_width/2:.1f}" y="{y - 10:.1f}" text-anchor="middle" font-family="Arial" font-size="13" fill="#222">{svg_text(value_label)}</text>',
            f'<text x="{x + bar_width/2:.1f}" y="{height - margin_bottom + 28}" text-anchor="middle" font-family="Arial" font-size="14" fill="#222">{svg_text(label)}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_horizontal_bar_chart(data, label_col, value_col, title, path, color="#4b8f69", width=980, height=640):
    margin_left, margin_right, margin_top, margin_bottom = 210, 80, 75, 45
    plot_width = width - margin_left - margin_right
    row_height = (height - margin_top - margin_bottom) / len(data)
    max_value = max(float(v) for v in data[value_col])
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">{svg_text(title)}</text>',
    ]
    for i, row in data.reset_index(drop=True).iterrows():
        value = float(row[value_col])
        bar_width = 0 if max_value == 0 else value / max_value * plot_width
        y = margin_top + i * row_height
        parts.extend([
            f'<text x="{margin_left - 12}" y="{y + row_height*0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#222">{svg_text(row[label_col])}</text>',
            f'<rect x="{margin_left}" y="{y + row_height*0.18:.1f}" width="{bar_width:.1f}" height="{row_height*0.58:.1f}" rx="3" fill="{color}"/>',
            f'<text x="{margin_left + bar_width + 8:.1f}" y="{y + row_height*0.62:.1f}" font-family="Arial" font-size="12" fill="#222">{int(value):,}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_length_box_svg(length_table, path, width=920, height=460):
    colors = {"Negative": "#b4494b", "Neutral": "#607d8b", "Positive": "#3f8f61"}
    axis_max = max(60, int(math.ceil(length_table["model_token_p95"].max() / 10) * 10))
    margin_left, margin_right, margin_top, margin_bottom = 150, 70, 80, 70
    plot_width = width - margin_left - margin_right
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Tweet Length By Sentiment</text>',
        f'<text x="{width/2}" y="{height-20}" text-anchor="middle" font-family="Arial" font-size="14" fill="#333">Model token count</text>',
    ]
    for tick in range(0, axis_max + 1, 10):
        x = margin_left + tick / axis_max * plot_width
        parts.append(f'<line x1="{x:.1f}" y1="{margin_top-8}" x2="{x:.1f}" y2="{height-margin_bottom+8}" stroke="#e0e0e0"/>')
        parts.append(f'<text x="{x:.1f}" y="{height-margin_bottom+28}" text-anchor="middle" font-family="Arial" font-size="12" fill="#555">{tick}</text>')
    for i, row in length_table.reset_index(drop=True).iterrows():
        y = margin_top + 70 * i
        sentiment = row["sentiment"]
        color = colors.get(sentiment, "#555")
        x25 = margin_left + row["model_token_p25"] / axis_max * plot_width
        x50 = margin_left + row["model_token_median"] / axis_max * plot_width
        x75 = margin_left + row["model_token_p75"] / axis_max * plot_width
        x90 = margin_left + row["model_token_p90"] / axis_max * plot_width
        x95 = margin_left + row["model_token_p95"] / axis_max * plot_width
        parts.extend([
            f'<text x="{margin_left-20}" y="{y+7}" text-anchor="end" font-family="Arial" font-size="15" font-weight="700" fill="#222">{svg_text(sentiment)}</text>',
            f'<line x1="{x25:.1f}" y1="{y}" x2="{x95:.1f}" y2="{y}" stroke="{color}" stroke-width="3"/>',
            f'<rect x="{x25:.1f}" y="{y-15}" width="{max(x75-x25, 1):.1f}" height="30" fill="{color}" opacity="0.35" stroke="{color}"/>',
            f'<line x1="{x50:.1f}" y1="{y-18}" x2="{x50:.1f}" y2="{y+18}" stroke="{color}" stroke-width="4"/>',
            f'<line x1="{x90:.1f}" y1="{y-13}" x2="{x90:.1f}" y2="{y+13}" stroke="{color}" stroke-width="2" stroke-dasharray="4 3"/>',
            f'<line x1="{x95:.1f}" y1="{y-13}" x2="{x95:.1f}" y2="{y+13}" stroke="{color}" stroke-width="2"/>',
            f'<text x="{x50:.1f}" y="{y+38}" text-anchor="middle" font-family="Arial" font-size="11" fill="#333">median {row["model_token_median"]:.0f}</text>',
            f'<text x="{x95:.1f}" y="{y-24}" text-anchor="middle" font-family="Arial" font-size="11" fill="#333">p95 {row["model_token_p95"]:.0f}</text>',
        ])
    save_svg("\n".join(parts), path, width, height)


def save_word_cloud_svg(token_table, title, path, color_palette, width=1000, height=620):
    top_words = token_table.head(70).copy()
    if top_words.empty:
        save_svg("", path, width, height)
        return
    min_count = top_words["count"].min()
    max_count = top_words["count"].max()
    x, y = 45, 92
    line_height = 0
    parts = [
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width/2}" y="38" text-anchor="middle" font-family="Arial" font-size="25" font-weight="700" fill="#222">{svg_text(title)}</text>',
    ]
    for i, row in top_words.reset_index(drop=True).iterrows():
        if max_count == min_count:
            font_size = 28
        else:
            scaled = (math.sqrt(row["count"]) - math.sqrt(min_count)) / (math.sqrt(max_count) - math.sqrt(min_count))
            font_size = 15 + scaled * 48
        word = str(row["token"])
        estimated_width = len(word) * font_size * 0.58
        if x + estimated_width > width - 45:
            x = 45
            y += line_height + 16
            line_height = 0
        if y > height - 35:
            break
        color = color_palette[i % len(color_palette)]
        parts.append(
            f'<text x="{x:.1f}" y="{y:.1f}" font-family="Arial" font-size="{font_size:.1f}" font-weight="700" fill="{color}">{svg_text(word)}</text>'
        )
        x += estimated_width + 24
        line_height = max(line_height, font_size)
    save_svg("\n".join(parts), path, width, height)


def display_svg_file(path):
    try:
        from IPython.display import SVG
        display(SVG(filename=str(path)))
    except Exception:
        print(path)


print("Phase 5 SVG helper functions ready.")

In [ ]:
# Generate Phase 5 figures.
phase5_figure_paths = {
    "sentiment_distribution": FIGURES_DIR / "phase5_sentiment_distribution.svg",
    "top_entities": FIGURES_DIR / "phase5_top_entities.svg",
    "length_by_sentiment": FIGURES_DIR / "phase5_tweet_length_by_sentiment.svg",
    "negative_word_cloud": FIGURES_DIR / "phase5_wordcloud_negative.svg",
    "positive_word_cloud": FIGURES_DIR / "phase5_wordcloud_positive.svg",
    "top_tokens_by_sentiment": FIGURES_DIR / "phase5_top_tokens_by_sentiment.svg",
}

save_vertical_bar_chart(
    sentiment_distribution_eda,
    label_col="sentiment",
    value_col="count",
    title="Sentiment Distribution",
    y_label="Tweet count",
    path=phase5_figure_paths["sentiment_distribution"],
    color="#2f6f9f",
)

save_horizontal_bar_chart(
    entity_distribution_eda.head(15),
    label_col="entity",
    value_col="count",
    title="Top 15 Entities In Cleaned Dataset",
    path=phase5_figure_paths["top_entities"],
    color="#5a6f8f",
)

save_length_box_svg(length_by_sentiment_eda, phase5_figure_paths["length_by_sentiment"])

save_word_cloud_svg(
    top_tokens_lookup["Negative"],
    "Negative Tweet Word Cloud",
    phase5_figure_paths["negative_word_cloud"],
    color_palette=["#7f1d1d", "#b4494b", "#6b2d2d", "#993f3f", "#333333"],
)

save_word_cloud_svg(
    top_tokens_lookup["Positive"],
    "Positive Tweet Word Cloud",
    phase5_figure_paths["positive_word_cloud"],
    color_palette=["#1f6f43", "#3f8f61", "#2f6f9f", "#4f7f50", "#333333"],
)

# Top-token chart uses the top 10 per sentiment for readability.
top_10_token_panels = phase5_top_tokens_by_sentiment.groupby("sentiment").head(10).copy()
panel_parts = ['<rect width="100%" height="100%" fill="#ffffff"/>']
panel_width, panel_height = 1080, 900
panel_parts.append('<text x="540" y="36" text-anchor="middle" font-family="Arial" font-size="24" font-weight="700" fill="#222">Top Analysis Tokens By Sentiment</text>')
panel_colors = {"Negative": "#b4494b", "Neutral": "#607d8b", "Positive": "#3f8f61"}
for panel_index, sentiment in enumerate(["Negative", "Neutral", "Positive"]):
    panel = top_10_token_panels[top_10_token_panels["sentiment"].eq(sentiment)].reset_index(drop=True)
    y_base = 70 + panel_index * 265
    max_count = panel["count"].max()
    panel_parts.append(f'<text x="40" y="{y_base}" font-family="Arial" font-size="20" font-weight="700" fill="#222">{sentiment}</text>')
    for i, row in panel.iterrows():
        y = y_base + 28 + i * 22
        bar_width = row["count"] / max_count * 600
        panel_parts.append(f'<text x="210" y="{y+13}" text-anchor="end" font-family="Arial" font-size="12" fill="#222">{svg_text(row["token"])}</text>')
        panel_parts.append(f'<rect x="225" y="{y}" width="{bar_width:.1f}" height="16" rx="3" fill="{panel_colors[sentiment]}"/>')
        panel_parts.append(f'<text x="{230 + bar_width:.1f}" y="{y+13}" font-family="Arial" font-size="12" fill="#222">{int(row["count"]):,}</text>')
save_svg("\n".join(panel_parts), phase5_figure_paths["top_tokens_by_sentiment"], panel_width, panel_height)

for figure_name, figure_path in phase5_figure_paths.items():
    print(f"{figure_name}: {figure_path}")

In [ ]:
# Save Phase 5 tables and written insights.
phase5_tables = {
    "phase5_basic_statistics.csv": phase5_basic_statistics,
    "phase5_sentiment_distribution.csv": sentiment_distribution_eda,
    "phase5_entity_distribution.csv": entity_distribution_eda,
    "phase5_length_by_sentiment.csv": length_by_sentiment_eda,
    "phase5_top_tokens_by_sentiment.csv": phase5_top_tokens_by_sentiment,
}

for filename, table in phase5_tables.items():
    table.to_csv(TABLES_DIR / filename, index=False)

negative_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Negative"), "count"].iat[0])
positive_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Positive"), "count"].iat[0])
neutral_count = int(sentiment_distribution_eda.loc[sentiment_distribution_eda["sentiment"].eq("Neutral"), "count"].iat[0])
top_entity = entity_distribution_eda.iloc[0]
positive_top_words = ", ".join(top_tokens_lookup["Positive"].head(8)["token"].tolist())
negative_top_words = ", ".join(top_tokens_lookup["Negative"].head(8)["token"].tolist())
neutral_top_words = ", ".join(top_tokens_lookup["Neutral"].head(8)["token"].tolist())

phase5_insights_lines = [
    "# Phase 5 EDA Insights",
    "",
    f"- The cleaned dataset contains {len(df_eda):,} tweets across {df_eda['entity'].nunique():,} entities and three sentiment labels.",
    f"- Sentiment distribution is moderately imbalanced: Negative {negative_count:,}, Positive {positive_count:,}, and Neutral {neutral_count:,}.",
    f"- Negative is the largest class, so later modeling should use stratified splits and report macro F1 in addition to accuracy.",
    f"- The most frequent entity is {top_entity['entity']} with {int(top_entity['count']):,} rows ({float(top_entity['percent']):.2f}%). Entity counts are fairly spread across the 32 topics.",
    f"- Model token lengths are short enough for sequence modeling: p95 is {df_eda['model_token_count'].quantile(0.95):.0f} tokens and p99 is {df_eda['model_token_count'].quantile(0.99):.0f} tokens, supporting a later padding length of about 60.",
    f"- Top negative analysis tokens include: {negative_top_words}.",
    f"- Top positive analysis tokens include: {positive_top_words}.",
    f"- Top neutral analysis tokens include: {neutral_top_words}.",
    "- The duplicate-text audit from Phase 4 remains important for model splitting, because repeated cleaned text can inflate performance if similar examples appear in both train and test sets.",
    "",
    "## Generated Figures",
    "",
]
phase5_insights_lines.extend([f"- {name}: {path}" for name, path in phase5_figure_paths.items()])
phase5_insights_lines.extend(
    [
        "",
        "## Modeling Implications",
        "",
        "- Use stratified splitting to preserve sentiment proportions.",
        "- Build the model vocabulary only on the training split to avoid leakage.",
        "- Treat 60 tokens as the initial max sequence length candidate for RNN padding.",
        "- Consider deduplicating repeated model_text and sentiment pairs before final modeling experiments if validation leakage appears likely.",
    ]
)

phase5_insights_path = REPORTS_DIR / "phase5_eda_insights.md"
phase5_insights_path.write_text("\n".join(phase5_insights_lines) + "\n", encoding="utf-8")

print(f"Saved {len(phase5_tables)} Phase 5 EDA tables to {TABLES_DIR}")
print(f"Saved Phase 5 EDA insights to {phase5_insights_path}")

In [ ]:
# Render figures inline when the notebook is opened in Jupyter.
for figure_path in phase5_figure_paths.values():
    display_svg_file(figure_path)

## Phase 5 Findings To Carry Forward

EDA confirms that the cleaned three-class dataset is suitable for sequence modeling, with moderate class imbalance and short tweet lengths.

Carry-forward points:

- Use stratified train, validation, and test splits.
- Use macro F1 during evaluation because the classes are not perfectly balanced.
- Start RNN sequence padding at 60 tokens, based on the Phase 4 and Phase 5 token-length distributions.
- Build the vocabulary from training data only.
- Keep the Phase 4 duplicate-text audit in mind before final modeling, because repeated cleaned text can create optimistic test results if split randomly without care.

# Part 3: RNN Model Development

Planned work:

- Create train, validation, and test splits.
- Build vocabulary from training data only.
- Convert tweets to padded token sequences.
- Train an embedding-based LSTM or GRU model.
- Use dropout and controlled regularization.
- Track learning curves across epochs.

# Part 4: Evaluation, Improvement, And Presentation

Planned work:

- Report accuracy, precision, recall, and F1-score.
- Plot confusion matrix and learning curves.
- Compare baseline and improved RNN models.
- Run a small hyperparameter search.
- Demonstrate predictions on sample tweets.
- Summarize findings in a presentation-ready format.